In [ ]:
!nvidia-smi


In [ ]:
%pip install -q pytorch-lightning==2.6.0 asteroid==0.7.0 flow-matching==1.0.10 einops==0.8.2 pesq pystoi torchdiffeq


In [ ]:
!git clone https://github.com/aleXiehta/AD-FlowTSE.git /content/AD-FlowTSE
%cd /content/AD-FlowTSE


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
ckpt_src = '/content/drive/MyDrive/t_predictor_noisy.ckpt'
ckpt_dst = '/content/AD-FlowTSE/t_predictor_noisy.ckpt'

if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, ckpt_dst)
    size_mb = os.path.getsize(ckpt_dst) / 1e6
    print(f"Copied checkpoint: {size_mb:.1f} MB")
else:
    print(f"NOT FOUND: {ckpt_src}")
    print("Please upload t_predictor_noisy.ckpt to your Google Drive root folder first!")


In [ ]:
import os, shutil

drive_data = '/content/drive/MyDrive/librimix_data'
local_data = '/content/librimix_data'
librimix_base = os.path.join(local_data, 'Libri2Mix/wav16k/min')

if os.path.exists(os.path.join(drive_data, 'Libri2Mix/wav16k/min/train-100')):
    print("Found cached LibriMix data on Google Drive! Copying to local disk...")
    !cp -r {drive_data} {local_data}
    print("Done copying from Drive.")
else:
    print("No cached data on Drive. Generating LibriMix from scratch...")

    if not os.path.exists('/content/LibriMix'):
        !git clone https://github.com/JorisCos/LibriMix.git /content/LibriMix

    %cd /content/LibriMix
    !bash generate_librimix.sh {local_data}

    print("\nSaving generated data to Google Drive (so you don't redo this)...")
    !cp -r {local_data} {drive_data}
    print("Saved to Google Drive!")

%cd /content/AD-FlowTSE

if not os.path.exists('/content/LibriMix'):
    !git clone https://github.com/JorisCos/LibriMix.git /content/LibriMix

print("\nGenerated data:")
for d in ['train-100', 'train-360', 'dev', 'test']:
    path = os.path.join(librimix_base, d)
    if os.path.exists(path):
        s1_dir = os.path.join(path, 'sep_noisy', 's1')
        n = len(os.listdir(s1_dir)) if os.path.exists(s1_dir) else '?'
        print(f"  {d}: {n} samples")
    else:
        print(f"  {d}: NOT FOUND")


In [ ]:
%cd /content/AD-FlowTSE

training_script = r'''
"""
Fine-tune T-Predictor with target-absent training examples.
"""

import argparse
import os
import yaml
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor,
)
from pytorch_lightning.loggers import TensorBoardLogger
from asteroid.engine.optimizers import make_optimizer
from torch.optim.lr_scheduler import (
    CosineAnnealingLR,
    LambdaLR,
    SequentialLR,
)

from flow_matching.path import CondOTProbPath
from data.datasets import get_dataloaders
from models.t_predicter import TPredicter


class TargetAbsentLightningModule(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.model = TPredicter(**config["model"])
        self.config = config
        self.save_hyperparameters(config)
        self.target_absent_prob = config["train"].get("target_absent_prob", 0.3)

    def forward(self, x, enrollment):
        return self.model(x, enrollment, aug=False)

    def training_step(self, batch, batch_idx):
        source = batch["source_rescaled"]
        background = batch["background_rescaled"]
        enrollment = batch["enroll"]
        batch_size = source.size(0)
        path = CondOTProbPath()

        alpha = torch.rand((batch_size,), device=source.device)
        absent_mask = torch.rand((batch_size,), device=source.device) < self.target_absent_prob

        source_modified = source.clone()
        alpha_target = alpha.clone()
        if absent_mask.any():
            source_modified[absent_mask] = 0.0
            alpha_target[absent_mask] = 0.0

        path_sample = path.sample(t=alpha, x_0=background, x_1=source_modified)
        mixture = path_sample.x_t

        alpha_hat = self.model(
            mixture, enrollment,
            aug=self.config["train"].get("spectral_aug", False),
        )
        loss = torch.nn.functional.mse_loss(alpha_hat, alpha_target)

        self.log("train_loss", loss, on_step=True, on_epoch=True,
                 prog_bar=True, sync_dist=True, batch_size=batch_size)
        self.log("train_absent_frac", absent_mask.float().mean(),
                 on_step=False, on_epoch=True, batch_size=batch_size)
        return loss

    def validation_step(self, batch, batch_idx):
        source = batch["source_rescaled"]
        background = batch["background_rescaled"]
        enrollment = batch["enroll"]
        batch_size = source.size(0)
        path = CondOTProbPath()

        alpha = torch.rand((batch_size,), device=source.device)
        absent_mask = torch.rand((batch_size,), device=source.device) < self.target_absent_prob

        source_modified = source.clone()
        alpha_target = alpha.clone()
        if absent_mask.any():
            source_modified[absent_mask] = 0.0
            alpha_target[absent_mask] = 0.0

        path_sample = path.sample(t=alpha, x_0=background, x_1=source_modified)
        mixture = path_sample.x_t

        alpha_hat = self.model(mixture, enrollment, aug=False)
        loss = torch.nn.functional.mse_loss(alpha_hat, alpha_target)

        self.log("val_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, sync_dist=True, batch_size=batch_size)
        return loss

    def configure_optimizers(self):
        optimizer = make_optimizer(self.model.parameters(), **self.config["optim"])
        sched_cfg = self.config["scheduler"]
        if sched_cfg["type"] == "CosineAnnealingLR":
            warmup = LambdaLR(
                optimizer,
                lr_lambda=lambda ep: ep / max(sched_cfg["warmup_epochs"], 1)
                if ep < sched_cfg["warmup_epochs"] else 1.0,
            )
            cosine = CosineAnnealingLR(
                optimizer, T_max=sched_cfg["t_max"], eta_min=sched_cfg["eta_min"],
            )
            return {
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": SequentialLR(
                        optimizer, schedulers=[warmup, cosine],
                        milestones=[sched_cfg["warmup_epochs"]],
                    ),
                    "interval": "epoch", "frequency": 1,
                },
            }
        return optimizer


class DataModule(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def setup(self, stage=None):
        self.train_loader, self.val_loader = get_dataloaders(
            self.config, is_ddp=False, world_size=1, rank=0
        )

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    with open(args.config, "r") as f:
        config = yaml.safe_load(f)

    pl.seed_everything(config["seed"])
    torch.set_float32_matmul_precision("medium")

    os.makedirs(config["train"]["log_dir"], exist_ok=True)
    os.makedirs(config["checkpoint"]["dir"], exist_ok=True)

    if config["checkpoint"]["load_weights_only"] and config["checkpoint"]["resume"]:
        model = TargetAbsentLightningModule.load_from_checkpoint(
            config["checkpoint"]["resume"], config=config, strict=True,
        )
        print(f"Loaded pretrained weights from {config['checkpoint']['resume']}")
    else:
        model = TargetAbsentLightningModule(config)

    data_module = DataModule(config)

    callbacks = []
    if config["early_stopping"]["enabled"]:
        callbacks.append(EarlyStopping(
            monitor=config["early_stopping"]["monitor"],
            patience=config["early_stopping"]["patience"],
            verbose=True, mode=config["early_stopping"]["mode"],
        ))
    callbacks.append(ModelCheckpoint(
        dirpath=config["checkpoint"]["dir"],
        filename=config["checkpoint"]["ckpt_name"],
        save_top_k=1, save_last=True, verbose=True,
        monitor=config["checkpoint"]["monitor"],
        mode=config["checkpoint"]["mode"],
    ))
    callbacks.append(LearningRateMonitor(logging_interval="epoch"))

    tb_logger = TensorBoardLogger(
        save_dir=config["train"]["log_dir"],
        name="lightning_logs", version="target_absent",
    )

    trainer = pl.Trainer(
        max_epochs=config["train"]["num_epochs"],
        accelerator="gpu", devices=1,
        accumulate_grad_batches=config["train"]["accumulation_steps"],
        callbacks=callbacks,
        default_root_dir=config["train"]["log_dir"],
        logger=tb_logger,
        log_every_n_steps=50,
        precision=config["train"]["precision"],
        gradient_clip_val=config["train"]["gradient_clip_val"],
    )

    trainer.fit(model, datamodule=data_module)
    print(f"\\nDone! Best checkpoint: {config['checkpoint']['dir']}")


if __name__ == "__main__":
    main()
'''

with open("train_t_predicter_target_absent.py", "w") as f:
    f.write(training_script)
print("Wrote train_t_predicter_target_absent.py")


In [ ]:
import yaml, os


librimix_base = "/content/librimix_data/Libri2Mix/wav16k/min"
metadata_dir = "/content/LibriMix/metadata"


for d in ["train-100", "train-360", "dev", "test"]:
    path = os.path.join(librimix_base, d)
    exists = os.path.exists(path)
    if exists:
        n_files = len(os.listdir(os.path.join(path, "sep_noisy", "s1"))) if os.path.exists(os.path.join(path, "sep_noisy", "s1")) else "?"
    else:
        n_files = 0
    print(f"  {d}: {'EXISTS' if exists else 'MISSING'} ({n_files} files)")

config = {
    "train": {
        "batch_size": 32,
        "num_epochs": 20,
        "accumulation_steps": 1,
        "num_workers": 4,
        "log_lr": True,
        "precision": "16-mixed",
        "log_dir": "exp/TPredictor_target_absent_finetune",
        "log_interval": 100,
        "detect_anomaly": False,
        "gradient_clip_val": 0.5,
        "limit_train_batches": None,
        "limit_val_batches": None,
        "overfit_batches": 0.0,
        "spectral_aug": True,
        "target_absent_prob": 0.3,
    },
    "scheduler": {
        "type": "CosineAnnealingLR",
        "t_max": 20,
        "eta_min": 0.000001,
        "warmup_epochs": 2,
        "lr_reduce_patience": 10,
        "lr_reduce_factor": 0.5,
        "start_epoch": 0,
    },
    "loss": {"gamma": 0.5},
    "model": {"C": 1024},
    "dataset": {
        "enable_train_360": os.path.exists(os.path.join(librimix_base, "train-360")),
        "enable_train_100": True,
        "train_360_dir": os.path.join(librimix_base, "train-360"),
        "train_100_dir": os.path.join(librimix_base, "train-100"),
        "val_dir": os.path.join(librimix_base, "dev"),
        "test_dir": os.path.join(librimix_base, "test"),
        "librimix_meta_dir": metadata_dir,
        "task": "sep_noisy",
        "n_src": 2,
        "sample_rate": 16000,
        "segment": 3,
        "segment_aux": 3,
        "n_fft": 510,
        "win_length": 510,
        "hop_length": 128,
        "snr_range": [-15, 15],
    },
    "optim": {
        "optimizer": "AdamW",
        "lr": 0.00002,
        "weight_decay": 0.01,
    },
    "ddp": {
        "use_ddp": False,
        "num_nodes": 1,
        "num_gpus": 1,
        "strategy": "ddp",
    },
    "seed": 42,
    "early_stopping": {
        "enabled": True,
        "monitor": "val_loss",
        "patience": 5,
        "verbose": True,
        "mode": "min",
        "delta": 0.0,
    },
    "checkpoint": {
        "dir": "exp/TPredictor_target_absent_finetune/checkpoints",
        "ckpt_name": "best",
        "save_best": True,
        "save_last": True,
        "verbose": True,
        "monitor": "val_loss",
        "mode": "min",
        "load_weights_only": True,
        "resume": "t_predictor_noisy.ckpt",
    },
    "eval": {
        "checkpoint": "exp/TPredictor_target_absent_finetune/checkpoints/best.ckpt",
    },
}

os.makedirs("config", exist_ok=True)
with open("config/config_finetune_colab.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
print("Wrote config/config_finetune_colab.yaml")


In [ ]:
!python train_t_predicter_target_absent.py --config config/config_finetune_colab.yaml


In [ ]:
import shutil, os

drive_out = '/content/drive/MyDrive/AD-FlowTSE_checkpoints'
os.makedirs(drive_out, exist_ok=True)

best_ckpt = "exp/TPredictor_target_absent_finetune/checkpoints/best.ckpt"
last_ckpt = "exp/TPredictor_target_absent_finetune/checkpoints/last.ckpt"

for src, name in [(best_ckpt, "t_predictor_target_absent_best.ckpt"),
                  (last_ckpt, "t_predictor_target_absent_last.ckpt")]:
    if os.path.exists(src):
        dst = os.path.join(drive_out, name)
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f"Saved {name} to Google Drive ({size_mb:.1f} MB)")
    else:
        print(f"Not found: {src}")

print(f"\nCheckpoints saved to: {drive_out}")
print("Download them to your local AD-FlowTSE/ folder when ready.")
